In [15]:
import duckdb
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 50)
con = duckdb.connect("../data/lending.duckdb")

הצגת עד 50 עמודות בטבלה והתחברות למסד הנתונים

In [16]:
con.sql(r"""
CREATE OR REPLACE TABLE loans_features AS
SELECT
    is_default,
    issue_year,
    strptime(issue_d, '%b-%Y') AS issue_date,
    loan_amnt,
    CAST(regexp_extract(term, '(\d+)', 1) AS INTEGER) AS term_months,
    int_rate,
    installment,
    grade,
    sub_grade,

    CASE
        WHEN emp_length IS NULL THEN NULL
        WHEN emp_length LIKE '< 1%' THEN 0
        WHEN emp_length LIKE '10+%' THEN 10
        ELSE CAST(regexp_extract(emp_length, '(\d+)', 1) AS INTEGER)
    END AS emp_years,

    home_ownership, 
    verification_status, 
    purpose, 
    addr_state, 
    application_type,
    annual_inc,
    
    loan_amnt / NULLIF(annual_inc, 0) AS loan_to_income,
    installment * 12 / NULLIF(annual_inc, 0) AS payment_to_income,

    dti,
    delinq_2yrs, 
    inq_last_6mths, 
    open_acc, 
    pub_rec, 
    revol_bal,
    revol_util, 
    total_acc, 
    mort_acc, 
    pub_rec_bankruptcies,

    (fico_range_low + fico_range_high) / 2.0 AS fico_avg,

    date_diff('month', 
    strptime(earliest_cr_line, '%b-%Y'),
    strptime(issue_d, '%b-%Y')) 
    / 12.0 AS credit_history_years

FROM loans
""")

con.sql("SELECT * FROM loans_features LIMIT 5").df()

,is_default,issue_year,issue_date,loan_amnt,term_months,int_rate,installment,grade,sub_grade,emp_years,home_ownership,verification_status,purpose,addr_state,application_type,annual_inc,loan_to_income,payment_to_income,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,fico_avg,credit_history_years
0,0,2015,2015-12-01,3600.0,36,13.99,123.03,C,C4,10,MORTGAGE,Not Verified,debt_consolidation,PA,Individual,55000.0,0.065455,0.026843,5.91,0.0,1.0,7.0,0.0,2765.0,29.7,13.0,1.0,0.0,677.0,12.333333
1,0,2015,2015-12-01,24700.0,36,11.99,820.28,C,C1,10,MORTGAGE,Not Verified,small_business,SD,Individual,65000.0,0.380000,0.151436,16.06,1.0,4.0,22.0,0.0,21470.0,19.2,38.0,4.0,0.0,717.0,16.000000
2,0,2015,2015-12-01,20000.0,60,10.78,432.66,B,B4,10,MORTGAGE,Not Verified,home_improvement,IL,Joint App,63000.0,0.317460,0.082411,10.78,0.0,0.0,6.0,0.0,7869.0,56.2,18.0,5.0,0.0,697.0,15.333333
3,0,2015,2015-12-01,10400.0,60,22.45,289.91,F,F1,3,MORTGAGE,Source Verified,major_purchase,PA,Individual,104433.0,0.099585,0.033312,25.37,1.0,3.0,12.0,0.0,21929.0,64.5,35.0,6.0,0.0,697.0,17.500000
4,0,2015,2015-12-01,11950.0,36,13.44,405.18,C,C3,4,RENT,Source Verified,debt_consolidation,GA,Individual,34000.0,0.351471,0.143005,10.20,0.0,0.0,5.0,0.0,8822.0,68.4,6.0,0.0,0.0,692.0,28.166667


is_default - חדלות פירעון
issue_year - שנת הנפקת ההלוואה
issue_d (Issue date) - תאריך הנפקת ההלוואה - טקסט
issue_date - תאריך הנפקת ההלוואה - תאריך
loan_amnt (Loan amount) - סכום ההלוואה
term - תקופת ההלוואה בחודשים - טקסט
term_months - תקופת ההלוואה בחודשים - מספר
int_rate (Interest Rate) - שיעור הריבית השנתית על ההלוואה
installment - תשלום חודשי עבור ההלוואה
grade - דירוג האשראי הכללי של ההלוואה
sub_grade - דירוג מפורט יותר בתוך הדירוג הכללי
emp_length (Employment length) - שנות ותק במקום עבודה - טקסט
emp_years (Employment years) - שנות ותק במקום עבודה - מספר
home_ownership - בעלות על בית
verification_status - אימות המידע הפיננסי
purpose - מטרת ההלוואה
addr_state (Address state) - מדינה בארה"ב
application_type - סוג הבקשה - אישית או משותפת עם לווה נוסף
annual_inc (Annual income) - הכנסה שנתית
loan_to_income - יחס סכום ההלוואה להכנסה שנתית
payment_to_income - יחס התשלום השנתי להכנסה השנתית
dti (Debt-to-Income ratio) - יחס חוב להכנסה - בודק כמה מההכנסה החודשית של הלווה מוקדשת להחזר חובות
הערה - החובות עשויות לכלול התחייבויות נוספות ולא רק את ההלוואה הנוכחית
delinq_2yrs (Delinquency in the last 2 years) - מספר הפיגורים בתשלומים בשנתיים האחרונות
inq_last_6mths (Inquiries in the last 6 months) - מספר בדיקות האשראי ב6 החודשים האחרונים
open_acc (Open accounts) - מספר חשבונות האשראי הפעילים
pub_rec (Public records) - מספר הרישומים הציבורים השליליים בדו"ח האשראי
revol_bal (Revolving balance) - יתרת החוב בחשבונות אשראי מתגלגלים
revol_util (Revolving utilization) - שיעור ניצול מסגרת האשראי
total_acc (Total accounts) - מספר חשבונות האשראי
mort_acc (Mortgage accounts) - מספר חשבונות המשכנתא
pub_rec_bankruptcies (Public records bankruptcies) - מספר רישומי פשיטת רגל
fico_range_low - הגבול התחתון של טווח דירוג האשראי
fico_range_high - הגבול העליון של טווח דירוג האשראי
fico_avg - ממוצע טווח דירוג האשראי
earliest_cr_line (Earliest credit line) - התאריך שבו נפתח חשבון האשראי הראשון של הלווה
credit_history_years - מספר שנות היסטוריית האשראי של הלווה

In [17]:
df = con.sql("SELECT * FROM loans_features").df()
print(df.shape)

df.groupby("issue_year")["is_default"].agg(loans="count", default_rate="mean").round(3)

(1345310, 30)


,loans,default_rate
issue_year,,
2007,251,0.179
2008,1562,0.158
2009,4716,0.126
2010,11536,0.129
2011,21721,0.152
2012,53367,0.162
2013,134804,0.156
2014,223102,0.184
2015,375545,0.202


חלוקת הנתונים לקבוצות לפי שנת מתן ההלוואה
מתמקדים רק בעמודה של חדלות פירעון
ספירת כמות הנתונים לכל קבוצה - ספירת כמות ההלוואות לכל שנת מתן ההלוואה
חישוב ממוצע הערכים לכל קבוצה - חישוב שיעור ההלוואות שהגיעו לחדלות פירעון
עיגול תוצאות החישובים ל3 ספרות אחרי הנקודה העשרונית

In [18]:
train = df[(df["issue_year"] >= 2009) & (df["issue_year"] <= 2014)].copy()
test  = df[df["issue_year"] == 2015].copy()

print(f"train: {len(train):,} loans, default rate {train['is_default'].mean():.3f}")
print(f"test:  {len(test):,} loans, default rate {test['is_default'].mean():.3f}")

train: 449,246 loans, default rate 0.170
test:  375,545 loans, default rate 0.202


חלוקה הנתונים לשתי קבוצות - קבוצת אימון וקבוצת בדיקה
בחירת הלוואות משנת 2009 ועד שנת 2014 לקבוצת האימון
בחירת הלוואות משנת 2015 בלבד לקבוצת בדיקה

In [19]:
num_cols = train.select_dtypes("number").columns.drop(["is_default", "issue_year"])

rows = []
for c in num_cols:
    s = train[[c, "is_default"]].dropna()
    auc = roc_auc_score(s["is_default"], s[c])
    rows.append((c, round(max(auc, 1 - auc), 3)))

pd.DataFrame(rows, columns=["feature", "single_feature_auc"]) \
  .sort_values("single_feature_auc", ascending=False)

,feature,single_feature_auc
2,int_rate,0.673
1,term_months,0.603
6,loan_to_income,0.600
18,fico_avg,0.588
7,payment_to_income,0.584
8,dti,0.569
5,annual_inc,0.554
0,loan_amnt,0.547
14,revol_util,0.547
10,inq_last_6mths,0.540


בחירת עמודות המכילות נתונים מספריים והשמטת העמודות חדלות פירעון ושנת הנפקת ההלוואה
בכל לולאה - בוחרים את העמודה הנוכחית ואת עמודת חדלות פירעון
ומסירים את השורות שבהן חסר ערך באחת משתי העמודות
מחשבים עד כמה העמודה הנוכחית (קטגוריה) מצליחה להבדיל בין לווים שיעמדו בתשלומי ההלוואה
לבין לווים שלא יעמדו בתשלומי ההלוואה
מוסיפים את החישוב לרשימה
לבסוף מציגים את החישובים בטבלה מובנית

In [20]:
con.close()

התנתקות ממסד הנתונים